# Hướng dẫn Benchmark MinHashLSH vs LSHBloom

## Giới thiệu

Notebook này giải thích **cách dữ liệu được sinh ra** cho benchmark khử trùng lặp nhằm so sánh **MinHashLSH** (tiêu chuẩn) vs **LSHBloom** (phương pháp mới).

### Nguồn dữ liệu gốc

- **AdaParse Dataset** (Siebenschuh et al., 2025)
- Chứa bài báo khoa học với 2 phiên bản:
  - Phiên bản HTML (từ web page)
  - Phiên bản PDF (từ file PDF, parse bằng 4 công cụ khác nhau)

### 4 Parser được sử dụng

| Parser | Mô tả |
|---|---|
| **HTML** | Trích xuất trực tiếp từ trang HTML |
| **PyMuPDF** | Thư viện Python để parse PDF |
| **Nougat** | Neural OCR cho Academic Documents |
| **Tesseract** | OCR engine kinh điển |

### Loại trùng lặp được mô phỏng

| Loại | Mô tả | Độ khó |
|---|---|---|
| **Parser Duplicate** (modification=1) | Cùng bài báo gốc, parse bằng công cụ khác nhau → Gần-trùng lặp do nhiễu parsing | Dễ - MinHash giỏi bắt |
| **Truncation Duplicate** (modification=2) | Cùng bài báo nhưng bị cắt xén 1-20% nội dung → Mô phỏng lỗi OCR | Khó - Similarity Jaccard thấp |

---

## Kiến trúc dữ liệu

```
AdaParse Dataset (gốc)
├── joint_to_html/parsed_pdfs/
│   └── *.jsonl    (mỗi dòng: {"path": "arxiv/pdf/1352.4534v1.pdf", "text": "..."})
├── joint_to_pymupdf/parsed_pdfs/
│   └── *.jsonl
├── joint_to_nougat/parsed_pdfs/
│   └── *.jsonl
└── joint_to_tesseract/parsed_pdfs/
    └── *.jsonl

↓ (Bước 1: Sample N_per_source từ mỗi parser)

Original Dataset
└── text | path | id | parser | modification=0

↓ (Bước 2: Collect cross-parser duplicates)

Parser Duplicates
└── [cùng PDF, khác parser] → modification=1

↓ (Bước 3: Create truncation duplicates)

Truncation Duplicates
└── [cắt xén 1-20%] → modification=2

↓ (Bước 4: Merge + Label)

Final Benchmark CSV
└── is_duplicate | modification | trunc_percentage | rel_location | ...
```

## Luồng sinh benchmark

### Bước 1: Lấy mẫu từ mỗi parser

In [ ]:
# Script gốc: synthetic_benchmark/gen_dedup_benchmark.py
# Không thay đổi - chỉ tham khảo

"""
    parser_sources = ['html', 'nougat', 'pymupdf', 'tesseract']
    N_per_source = 4000  # Lấy 4000 bài báo từ mỗi parser
    
    Quá trình:
    1. Duyệt từng parser folder (./joint_to_{parser}/parsed_pdfs/)
    2. Đọc các file JSONL và lấy N_per_source dòng
    3. Kiểm tra: text không rỗng + chưa được lấy bởi parser khác trước đó
    4. Result: ~16000 tài liệu gốc (4000 × 4 parser, nhưng có overlap do cùng PDF)
"""

print("Bước 1: Sample từ mỗi parser")
print("- html: 4000 bài báo từ ./joint_to_html/parsed_pdfs/*.jsonl")
print("- pymupdf: 4000 bài báo từ ./joint_to_pymupdf/parsed_pdfs/*.jsonl")
print("- nougat: 4000 bài báo từ ./joint_to_nougat/parsed_pdfs/*.jsonl")
print("- tesseract: 4000 bài báo từ ./joint_to_tesseract/parsed_pdfs/*.jsonl")
print()
print("Kết quả: data_list (N tài liệu gốc, modification=0)")

### Bước 2: Tìm cross-parser duplicates

In [ ]:
# Script gốc: dedup_benchmark_utils.py - collect_duplicates()
"""
    Logic:
    1. Từ mỗi parser, lấy 40 đường dẫn ("path") của bài báo đã sample
    2. Duyệt các parser khác, tìm xem đường dẫn này có tồn tại không
    3. Nếu tồn tại → marking as duplicate (cùng PDF, khác parser)
    4. Result: data_dupl_list (gần-trùng lặp do parser khác nhau)
"""

print("Bước 2: Collect Cross-Parser Duplicates")
print()
print("Ví dụ:")
print("  PDF: arxiv/pdf/1352.4534v1.pdf")
print("  - Phiên bản HTML: text_html")
print("  - Phiên bản PyMuPDF: text_pymupdf")
print("  - Phiên bản Nougat: text_nougat")
print()
print("Cùng PDF nhưng nội dung hơi khác (do parsing differences)")
print("  Jaccard similarity: ~0.8-0.95 (gần-trùng)")
print()
print("Kết quả: data_dupl_list (parser duplicates, modification=1)")

### Bước 3: Tạo truncation duplicates

In [ ]:
# Script gốc: dedup_benchmark_utils.py - truncate()
"""
    Cắt xén nội dung để mô phỏng lỗi OCR hoặc cắt bớt:
    
    Tham số:
    - percentage: {1, 2, 5, 7, 10, 15, 20}%
    - rel_location (vị trí): {0, 0.25, 0.5, 0.75, 1.0}
      * 0: cắt từ đầu
      * 0.25: cắt 1/4 vào
      * 0.5: cắt từ giữa
      * 0.75: cắt 3/4 vào
      * 1.0: cắt từ cuối
"""

print("Bước 3: Create Truncation Duplicates")
print()
print("Ví dụ cắt xén:")
print()
print("Original: 'The quick brown fox jumps over the lazy dog'")
print()
print("Cắt 20% từ giữa (rel_location=0.5):")
print("  → 'The quick brown fox jumps  lazy dog'")
print("  → Jaccard ~ 0.7 (khó hơn để detect)")
print()
print("Kết quả: data_trunc_list (truncation duplicates, modification=2)")

### Bước 4: Merge và labeling

In [ ]:
# Script gốc: dedup_benchmark_utils.py - make_benchmark_dataframe()
"""
    Merge 3 danh sách:
    1. data_list: tài liệu gốc
    2. data_dupl_list: parser duplicates
    3. data_trunc_list: truncation duplicates
    
    Tạo nhãn:
    - is_duplicate = 0: tài liệu gốc (modification=0)
    - is_duplicate = 1: duplicates (modification={1 hoặc 2})
"""

import pandas as pd

# Ví dụ cấu trúc DataFrame cuối cùng
example_df = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'text': ['document 1', 'document 2 (truncated)', 'document 3', 'document 4 (from pymupdf)', 'document 5'],
    'path': ['arxiv/1.pdf', 'arxiv/1.pdf', 'arxiv/2.pdf', 'arxiv/1.pdf', 'arxiv/3.pdf'],
    'parser': ['html', 'html', 'html', 'pymupdf', 'nougat'],
    'is_duplicate': [0, 1, 0, 1, 0],
    'modification': [0, 2, 0, 1, 0],
    'trunc_percentage': [0, 20, 0, 0, 0],
    'rel_location': [0, 0.5, 0, 0, 0]
})

print("Cấu trúc DataFrame cuối cùng:")
print(example_df)
print()
print("Giải thích:")
print("- ID 1: Gốc (html)")
print("- ID 2: Truncation dup của ID 1 (cắt 20% từ giữa)")
print("- ID 3: Gốc (html)")
print("- ID 4: Parser dup của ID 1 (cùng PDF, từ pymupdf)")
print("- ID 5: Gốc (nougat)")
print()
print("Tỷ lệ trùng lặp: 2/5 = 40%")

---

## Cách tải dữ liệu gốc

### Nếu bạn muốn tái hiện (reproduce) benchmark:

1. **Thêm dữ liệu gốc AdaParse**:
   - Tải từ: [Siebenschuh et al. AdaParse Dataset](https://github.com/allenai/AdaParse)
   - Hoặc yêu cầu từ tác giả bài báo LSHBloom

2. **Cấu trúc thư mục**:
   ```
   LSHBloomExperiments/
   ├── joint_to_html/parsed_pdfs/
   ├── joint_to_pymupdf/parsed_pdfs/
   ├── joint_to_nougat/parsed_pdfs/
   └── joint_to_tesseract/parsed_pdfs/
   ```

3. **Chạy script sinh benchmark**:
   ```bash
   python synthetic_benchmark/gen_dedup_benchmark.py -p 0.5 -o my_benchmark
   ```
   - `-p 0.5`: Tỷ lệ trùng lặp = 50%
   - `-o my_benchmark`: Tên output CSV
   - Output: `benchmark_dfs/my_benchmark.csv`

### Hoặc sử dụng benchmark có sẵn:

- Các tác giả thường cung cấp pre-built benchmark
- Hãy check phần release của repo gốc

## Chạy thực nghiệm

### 1. Sinh benchmark (nếu có dữ liệu gốc)

In [ ]:
# Chạy từ terminal:
# cd synthetic_benchmark
# python gen_dedup_benchmark.py -p 0.5 -o 50k_p0.5

# Hoặc từ notebook:
import subprocess
import os
os.chdir('synthetic_benchmark')
# result = subprocess.run(['python', 'gen_dedup_benchmark.py', '-p', '0.5', '-o', '50k_test'], capture_output=True, text=True)
# print(result.stdout)
# if result.stderr:
#     print("Error:", result.stderr)

print("⚠️ Chỉ chạy nếu bạn đã có dữ liệu gốc AdaParse")

### 2. Chuyển đổi sang JSONL

In [ ]:
# python synthetic_benchmark/make_jsonl.py
print("Chuyển CSV → JSONL cho khử trùng lặp")

### 3. Chạy các thuật toán khử trùng lặp

In [ ]:
# MinHashLSH (yêu cầu Redis)
# python dedup/lsh/lsh.py --input 50k_p0.5 --sim-threshold 0.5 --num-perm 256

# LSHBloom (không cần Redis)
# python dedup/lsh/lsh_bloom.py --input 50k_p0.5 --sim-threshold 0.5 --num-perm 256

# DCLM
# python dedup/dclm/dclm.py --input 50k_p0.5 --sim-threshold 0.2 --ngram-size 5

# Dolma
# python dedup/dolma/dolma.py --input 50k_p0.5 --sim-threshold 0.2

# CCNet
# python dedup/ccnet/ccnet.py --input 50k_p0.5 --sim-threshold 0.2

print("Các script khử trùng lặp - không thay đổi, chỉ tham khảo")

---

## Tóm tắt

| Bước | Input | Script | Output | Mục đích |
|---|---|---|---|---|
| 1 | AdaParse (4 parser) | `gen_dedup_benchmark.py` | CSV benchmark | Sinh dữ liệu |
| 2 | Benchmark CSV | `make_jsonl.py` | JSONL | Format cho dedup |
| 3 | JSONL | `lsh_bloom.py` | Predictions CSV | So sánh thuật toán |
| 4 | Predictions + Ground truth | `score()` | Metrics (F1, Recall...) | Đánh giá |

**Quan trọng**: Tất cả script gốc **KHÔNG được thay đổi** - chỉ tham khảo, giải thích, và tài liệu hóa trong notebook này.